In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import glob
import os
import numpy as np
import h5py
import matplotlib.colors as colors
import random

import sys, os
# This is not super pretty, but I think this is the best way to import stuff from ../../../util?
CODE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if CODE_ROOT not in sys.path:
    sys.path.insert(1, CODE_ROOT)

from util.sim_data_helpers import get_data_from_snap_folder, get_data_from_header, get_cosmo_parameters
from util.spectra_helpers import load_spectra_from_boxes

In [ ]:
def neutral_hydrogen_hist(path, snapN, resolution, little_h, _range=None):

    # load gas data
    coordinates = get_data_from_snap_folder(path, snapN, "PartType0", "Coordinates")
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    box_size = get_data_from_header(path, snapN, "BoxSize")  # ckpc/h

    physical_bin_volume = ((box_size/1e3)/resolution)**2 * (box_size/1e3)

    if _range is None:
        _range = [[0, box_size], [0, box_size]]

    # mass-weighted histogram
    h, xedges, yedges = np.histogram2d(
        coordinates[:,0],
        coordinates[:,1],
        bins=resolution,
        range=_range,
        weights=HI_mass
    )

    # convert to projected density
    h = h / physical_bin_volume

    return h, xedges, yedges


def neutral_hydrogen_mass(path, snapN, little_h):

    # load gas data
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    total_HI_mass = np.sum(HI_mass)

    return total_HI_mass


def ionized_hydrogen_mass(path, snapN, little_h):

    # load gas data
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    total_HI_mass = np.sum(HI_mass)
    total_H_mass = np.sum(masses * X_H)

    return total_H_mass - total_HI_mass


def neutral_hydrogen_mass_fraction(path, snapN, little_h):

    # load gas data
    masses = get_data_from_snap_folder(path, snapN, "PartType0", "Masses") * 1e10/little_h  # M_sun
    f_HI = get_data_from_snap_folder(path, snapN, "PartType0", "NeutralHydrogenAbundance")

    # neutral hydrogen mass per cell
    X_H = 0.76 # TODO: is this correct??
    HI_mass = masses * X_H * f_HI

    total_HI_mass = np.sum(HI_mass)
    total_H_mass = np.sum(masses * X_H)

    return (total_H_mass - total_HI_mass) / total_H_mass

In [ ]:
paths = ["/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/gaikwad_UVB/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/puchwein_UVB/"]
labels = ["gaikwad_UVB", "puchwein_UVB"]
snapN = 3

ref_path = "/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/"
ref_h, _, _ = neutral_hydrogen_hist(ref_path, snapN, 128, little_h=get_cosmo_parameters(ref_path)[3])

plot_list = []
for path in paths:
    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)
    h, xedges, yedges = neutral_hydrogen_hist(path, snapN, 128, little_h=HubbleParam)
    plot_list.append((h, xedges, yedges))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

box_size = get_data_from_header(path, snapN, "BoxSize")  # ckpc/h

for i, ax in enumerate(axes):
    h, xedges, yedges = plot_list[i]
    diff_h = h/ref_h
    im = ax.imshow(diff_h.T, origin='lower', cmap='bwr', extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], norm=colors.LogNorm())
    ax.set_xticklabels('')
    ax.set_yticklabels('')
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_title(labels[i])

    ax.set_facecolor('black')
    
    # Add white scale bar (1 Mpc/h)
    scale_length = 5000  # ckpc/h
    scale_pos_x = 0.05 * box_size
    scale_pos_y = 0.05 * box_size
    ax.add_patch(plt.Rectangle((scale_pos_x, scale_pos_y), scale_length, 0.005 * box_size, color='white'))
    ax.annotate('5 Mpc/h', (scale_pos_x + scale_length/2, scale_pos_y + 0.01 * box_size), color='white', ha='center', va='bottom')

# Add color bar
cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label('Delta HI Density (M_sun/Mpc^3)')

plt.show()

In [ ]:
paths = ["/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/gaikwad_UVB/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/puchwein_UVB/"]
labels = ["Reference", "gaikwad_UVB", "puchwein_UVB"]
snapNs = [1, 2, 3, 4, 5]

panels = []

for path in paths:
    if path == "/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/":
        snapNs = [1, 2, 3, 4]
    else:
        snapNs = [1, 2, 3, 4, 5]
    redshifts = []
    for snapN in snapNs:
        z = get_data_from_header(path, snapN, "Redshift")
        redshifts.append(z)

    Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)

    HII_fractions = []
    for snapN in snapNs:
        HII_fraction = neutral_hydrogen_mass_fraction(path, snapN, HubbleParam)
        HII_fractions.append(HII_fraction*100)  

    panels.append((redshifts, HII_fractions))


In [ ]:

for i in range(len(paths)):
    redshifts, HII_fractions = panels[i]
    plt.plot(redshifts, HII_fractions, marker='o', linestyle='--', label=labels[i])
plt.xlabel('Redshift')
plt.ylabel('HII fraction in %')
# plt.gca().invert_xaxis()
plt.legend()
plt.show()

In [ ]:
path = "/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/"
snapNs = [1, 2, 3, 4]

redshifts = []
for snapN in snapNs:
    z = get_data_from_header(path, snapN, "Redshift")
    redshifts.append(z)

print(redshifts)

Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(path)

HI_masses = []
for snapN in snapNs:
    HI_mass = ionized_hydrogen_mass(path, snapN, HubbleParam)
    HI_masses.append(HI_mass)  

plt.plot(redshifts, HI_masses, marker='o', linestyle='--')
plt.xlabel('Redshift')
plt.ylabel('Total HII Mass [M_sun]')
plt.gca().invert_xaxis()
plt.show()

In [ ]:
paths = ["/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/gaikwad_UVB/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/puchwein_UVB/"]
labels = ["Reference", "gaikwad_UVB", "puchwein_UVB"]

Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(paths[0])

metals_paths = [i + "output/txt-files/metals_tot.txt" for i in paths]

x_plot = []
y_plot = []
for metals_path in metals_paths:
    data = np.loadtxt(metals_path)

    column_to_plot = 1

    x = data[:, 0]   # x-axis values (can be modified as needed)
    y = data[:, column_to_plot] # * 1e10/HubbleParam   # chosen column as y-axis

    x = 1/x - 1

    x_plot.append(x)
    y_plot.append(y)

In [ ]:
# Create the plot
for i in range(len(paths)):
    x = x_plot[i]
    y = y_plot[i]

    plt.plot(x, y, label=labels[i])

plt.xlim(0, 20)
# Labels
plt.xlabel('Redshift?')
plt.ylabel(f'Column {column_to_plot}')
plt.title('Plot from metals_tot.txt')
plt.legend()
# Show plot
plt.show()

In [ ]:
paths = ["/vera/u/jerbo/my_ptmp/L25n128_suite/reference_point/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/gaikwad_UVB/", "/vera/u/jerbo/my_ptmp/L25n128_UVB_tests/puchwein_UVB/"]

w, y, p = load_spectra_from_boxes(paths, [i for i in range(10000)])

In [ ]:
fluxes_ref = np.array(y[0])
fluxes_gaikwad = np.array(y[1])
fluxes_puchwein = np.array(y[2])

In [ ]:
from util.cosmo_helpers import redshift_wavelength_forward
import temet

def get_edges(gp_path):
    Ly_alpha_0 = 1215.67 

    sim = temet.sim(gp_path, redshift=2.0)
    dz = sim.dz
    z = get_data_from_header(gp_path, 3, param="Redshift")
    z = z + dz
    right_edge = redshift_wavelength_forward(z, Ly_alpha_0)
    left_edge = redshift_wavelength_forward(z - dz, Ly_alpha_0)

    return left_edge, right_edge

In [ ]:
ref_l, ref_r = get_edges(paths[0])
gai_l, gai_r = get_edges(paths[1])
puc_l, puc_r = get_edges(paths[2])

fluxes_ref = fluxes_ref[:, (w[0] >= ref_l) & (w[0] <= ref_r)]
fluxes_gaikwad = fluxes_gaikwad[:, (w[1] >= gai_l) & (w[1] <= gai_r)]
fluxes_puchwein = fluxes_puchwein[:, (w[2] >= puc_l) & (w[2] <= puc_r)]

In [ ]:
print(fluxes_ref.mean(axis=1))

In [ ]:
range_flux = (0.7, 1.4)
h_ref, edges_ref = np.histogram(fluxes_ref.mean(axis=1), bins=50, range=range_flux)
h_gaikwad, edges_gaikwad = np.histogram(fluxes_gaikwad.mean(axis=1), bins=50, range=range_flux)
h_puchwein, edges_puchwein = np.histogram(fluxes_puchwein.mean(axis=1), bins=50, range=range_flux)

fig = plt.subplots(figsize=(4, 3))

# Simulated Spectra
plt.stairs(h_ref, edges_ref, fill=True, alpha=0.6, label=f"Reference dataset", color="#2ca02c")
plt.stairs(h_gaikwad, edges_gaikwad, fill=True, alpha=0.6, label=f"Gaikwad dataset", color="#1f77b4")
plt.stairs(h_puchwein, edges_puchwein, fill=True, alpha=0.6, label=f"Puchwein dataset", color="#ff7f0e")

plt.xlim(range_flux)
plt.xlabel("Mean Flux per spectrum")
plt.ylabel("Number of spectra")
plt.legend(fontsize=8)
plt.tight_layout()

plt.show()